In [ ]:
!pip install google-generativeai langchain langchain-core

In [5]:
# ==============================================
# 1️⃣ IMPORTS
# ==============================================
import google.generativeai as genai
from getpass import getpass
from langchain.output_parsers import CommaSeparatedListOutputParser, DatetimeOutputParser
import json
from datetime import datetime
import re
from langchain_core.output_parsers import StrOutputParser


In [6]:
parser = StrOutputParser()

In [2]:
# ==============================================
# 2️⃣ API KEY
# ==============================================
api_key = getpass("🔑 Enter your Gemini API key: ")
genai.configure(api_key=api_key)

# Initialize the model
model = genai.GenerativeModel('gemini-2.5-flash')

In [3]:
# ==============================================
# 3️⃣ HELPER: Call Gemini
# ==============================================
def call_gemini(prompt: str) -> str:
    response = model.generate_content(prompt)
    return response.text

In [7]:
# ==============================================
# 4️⃣ STRING OUTPUT PARSER
# ==============================================
print("----- STRING OUTPUT PARSER -----")
string_prompt = "Write a short greeting message for students learning LangChain."
response = call_gemini(string_prompt)
print("LLM Response:", response)
parsed_string = parser.parse(response)
print("Parsed Output:", parsed_string)

----- STRING OUTPUT PARSER -----
LLM Response: Here are a few options, choose the one that best fits your tone!

**Option 1 (Concise & Enthusiastic):**
"Welcome, future LangChain builders! Get ready to dive into the exciting world of orchestrating large language models and creating powerful AI applications. Your journey starts now!"

**Option 2 (Slightly more encouraging/warm):**
"Hello students! Welcome to your LangChain learning adventure. Prepare to discover how to seamlessly connect large language models and build intelligent, innovative applications. We're excited to see what you create!"

**Option 3 (Very brief):**
"Welcome to LangChain, students! Get ready to harness the power of LLMs and build amazing things."
Parsed Output: Here are a few options, choose the one that best fits your tone!

**Option 1 (Concise & Enthusiastic):**
"Welcome, future LangChain builders! Get ready to dive into the exciting world of orchestrating large language models and creating powerful AI applicati

In [8]:
# ==============================================
# 5️⃣ CSV / Comma-Separated List PARSER
# ==============================================
print("\n----- CSV OUTPUT PARSER -----")
csv_prompt = "Give me 5 programming languages separated by commas."
response = call_gemini(csv_prompt)
print("LLM Response:", response)
csv_parser = CommaSeparatedListOutputParser()
parsed_list = csv_parser.parse(response)
print("Parsed Output (List):", parsed_list)


----- CSV OUTPUT PARSER -----
LLM Response: Python, Java, C++, JavaScript, C#
Parsed Output (List): ['Python', 'Java', 'C++', 'JavaScript', 'C#']


In [9]:
# ==============================================
# 6️⃣ DATETIME OUTPUT PARSER
# ==============================================
print("\n----- DATETIME OUTPUT PARSER -----")
datetime_prompt = "Give today's date in full ISO 8601 format (YYYY-MM-DDTHH:MM:SS.sssZ)."
response = call_gemini(datetime_prompt)
print("LLM Response:", response)

datetime_parser = DatetimeOutputParser()
parsed_datetime = datetime_parser.parse(response)
print("Parsed Output:", parsed_datetime)


----- DATETIME OUTPUT PARSER -----
LLM Response: 2023-10-27T10:30:45.123Z
Parsed Output: 2023-10-27 10:30:45.123000


In [11]:
# ==============================================
# 7️⃣ COMBINED PARSERS WORKFLOW (Structured JSON)
# ==============================================
print("\n----- COMBINED PARSERS WORKFLOW (Structured JSON) -----")

# Prompt LLM to return JSON
combined_prompt = """
Return the following information as a valid JSON object with real values:
{
    "fruits": ["fruit1", "fruit2", "fruit3"],
    "event_date": "YYYY-MM-DD",
    "message": "short welcome message"
}

Make sure to:
- Replace "YYYY-MM-DD" with today's date in actual format, e.g., "2026-02-16".
- Replace placeholder fruits and message with real examples.
- Keys must be exactly: fruits, event_date, message
- Return ONLY the JSON object, no other text.
"""

response = call_gemini(combined_prompt)
print("LLM Raw Response:\n", response)

# Parse JSON
try:
    data = json.loads(response)
except json.JSONDecodeError:
    # Some LLMs may include extra text, extract JSON block
    match = re.search(r"\{.*\}", response, re.DOTALL)
    if match:
        data = json.loads(match.group())
    else:
        raise ValueError("Could not parse JSON from LLM response")

# Use LangChain parser for fruits (comma-separated)
fruits = CommaSeparatedListOutputParser().parse(", ".join(data["fruits"]))

# Parse date manually (YYYY-MM-DD)
event_date = datetime.strptime(data["event_date"], "%Y-%m-%d")

# Message is just a string
message = data["message"]

print("Parsed Fruits:", fruits)
print("Parsed Event Date:", event_date)
print("Parsed Message:", message)


----- COMBINED PARSERS WORKFLOW (Structured JSON) -----
LLM Raw Response:
 ```json
{
    "fruits": ["apple", "banana", "orange"],
    "event_date": "2024-07-28",
    "message": "Welcome to our event! We're glad you're here."
}
```
Parsed Fruits: ['apple', 'banana', 'orange']
Parsed Event Date: 2024-07-28 00:00:00
Parsed Message: Welcome to our event! We're glad you're here.
